In [2]:
!pip install -q sentence-transformers faiss-cpu pandas numpy

In [3]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np

c:\medication-recovery\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

model = SentenceTransformer(MODEL_NAME)

print("Model loaded successfully!")

c:\medication-recovery\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3061.41it/s]


Model loaded successfully!


In [7]:
import pandas as pd

df = pd.read_csv("../data/raw/updated_indian_medicine_data.csv", on_bad_lines='skip')

print(df.columns.tolist())
print(df.head())

['id', 'name', 'price', 'Is_discontinued', 'manufacturer_name', 'type', 'pack_size_label', 'short_composition1', 'short_composition2', 'salt_composition', 'medicine_desc', 'side_effects', 'drug_interactions']
   id                      name   price  Is_discontinued  \
0   1  Augmentin 625 Duo Tablet  223.42            False   
1   2       Azithral 500 Tablet  132.36            False   
2   3          Ascoril LS Syrup  118.00            False   
3   4      Allegra 120mg Tablet  218.81            False   
4   5            Avil 25 Tablet   10.96            False   

                      manufacturer_name       type         pack_size_label  \
0  Glaxo SmithKline Pharmaceuticals Ltd  allopathy     strip of 10 tablets   
1           Alembic Pharmaceuticals Ltd  allopathy      strip of 5 tablets   
2          Glenmark Pharmaceuticals Ltd  allopathy  bottle of 100 ml Syrup   
3                     Sanofi India  Ltd  allopathy     strip of 10 tablets   
4                     Sanofi India  Ltd 

In [8]:
df["name"] = df["name"].fillna("")
df["salt_composition"] = df["salt_composition"].fillna("")
df["medicine_desc"] = df["medicine_desc"].fillna("")

In [9]:
df["embedding_text"] = (
    "Medicine Name: " + df["name"] +
    "\n\nComposition: " + df["salt_composition"] +
    "\n\nDescription: " + df["medicine_desc"]
)

In [10]:
print(df["embedding_text"].iloc[0])

Medicine Name: Augmentin 625 Duo Tablet

Composition: Amoxycillin  (500mg) +  Clavulanic Acid (125mg)

Description: Augmentin 625 Duo Tablet is a penicillin-type of antibiotic that helps your body fight infections caused by bacteria. It is used to treat infections of the lungs (e.g., pneumonia), ear, nasal sinus, urinary tract, skin and soft tissue. It will not work for viral infections such as the common cold.Augmentin 625 Duo Tablet is best taken with a meal to reduce the chance of a stomach upset. You should take it regularly at evenly spaced intervals as per the schedule prescribed by your doctor. Taking it at the same time every day will help you to remember to take it. The dose will depend on what you are being treated for, but you should always complete a full course of this antibiotic as prescribed by your doctor. Do not stop taking it until you have finished, even when you feel better. If you stop taking it early, some bacteria may survive and the infection may come back or wo

In [11]:
embeddings = model.encode(
    df["embedding_text"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=64
)

print(embeddings.shape)

Batches:  11%|█         | 422/3969 [08:20<1:10:10,  1.19s/it]


KeyboardInterrupt: 

In [10]:
print(type(embeddings))
print(embeddings.dtype)
print(embeddings.shape)

<class 'numpy.ndarray'>
float32
(253973, 384)


In [11]:
# Example transcript (replace this with one from your dataset)
transcript = """
Good afternoon, welcome to our clinic.
What brings you in today? My sugar levels have been higher than usual.
 Has your doctor prescribed any medicine? Oh yeah,
 I'm taking MeliGlyp M1 tablet SR as advised by my doctor
"""

# Encode the transcript
transcript_embedding = model.encode(
    transcript,
    convert_to_numpy=True
)

print(type(transcript_embedding))
print(transcript_embedding.dtype)
print(transcript_embedding.shape)

<class 'numpy.ndarray'>
float32
(384,)


In [12]:
print(df.shape)

(253973, 14)


In [13]:
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
similarities = cosine_similarity(
    transcript_embedding.reshape(1, -1),
    embeddings
)

In [15]:
print(similarities.shape)

(1, 253973)


In [16]:
import numpy as np

top_k = 5

top_indices = np.argsort(similarities[0])[-top_k:][::-1]

print(top_indices)

[100304  93993  98117 199652 100147]


In [17]:
df.iloc[top_indices][["name", "salt_composition"]]

,name,salt_composition
100304,Glucoster M 80mg/500mg Tablet,
93993,Flagday 50mg Syrup,
98117,Glucoway G 2mg/500mg Tablet,
199652,Sugar Control 1mg/500mg Tablet SR,
100147,Glucoact-M 80mg/500mg Tablet,


In [18]:
Meliglip_index = df[df["name"].str.contains("Meliglip", case=False, na=False)].index

print(Meliglip_index)

Index([144892, 145341], dtype='int64')


In [19]:
Meliglip_idx = Meliglip_index[0]

rank = np.where(np.argsort(similarities[0])[::-1] == Meliglip_idx)[0][0] + 1

print(rank)

13527


In [20]:
df["embedding_text_name"] = "Medicine Name: " + df["name"].fillna("")

In [21]:
embeddings_name = model.encode(
    df["embedding_text_name"].tolist(),
    convert_to_numpy=True,
    batch_size=64,
    show_progress_bar=True
)

Batches:   0%|          | 0/3969 [00:00<?, ?it/s]

In [22]:
similarities_name = cosine_similarity(
    transcript_embedding.reshape(1, -1),
    embeddings_name
)

In [23]:
top_k = 5

top_indices_name = np.argsort(similarities_name[0])[-top_k:][::-1]

df.iloc[top_indices_name][["name", "salt_composition"]]

,name,salt_composition
93993,Flagday 50mg Syrup,
99902,Glucoway VG 1mg/500mg/0.2mg Tablet,
100304,Glucoster M 80mg/500mg Tablet,
98117,Glucoway G 2mg/500mg Tablet,
199652,Sugar Control 1mg/500mg Tablet SR,


In [24]:
Meliglip_index = df[df["name"].str.contains("Meliglip", case=False, na=False)].index

Meliglip_idx = Meliglip_index[0]

rank = np.where(
    np.argsort(similarities_name[0])[::-1] == Meliglip_idx
)[0][0] + 1

print(rank)

9779


In [25]:
df.iloc[top_indices_name][["name", "medicine_desc"]]

,name,medicine_desc
93993,Flagday 50mg Syrup,
99902,Glucoway VG 1mg/500mg/0.2mg Tablet,
100304,Glucoster M 80mg/500mg Tablet,
98117,Glucoway G 2mg/500mg Tablet,
199652,Sugar Control 1mg/500mg Tablet SR,


In [26]:
medicine_utterance = """
I'm taking chemical SP 100 mg 325 mg as advised by my doctor."""

query_embedding = model.encode(
    medicine_utterance,
    convert_to_numpy=True
)

In [27]:
print(type(query_embedding))
print(query_embedding.dtype)
print(query_embedding.shape)

<class 'numpy.ndarray'>
float32
(384,)


In [28]:
similarities_query = cosine_similarity(
    query_embedding.reshape(1, -1),
    embeddings_name
)

In [29]:
top_k = 5

top_indices = np.argsort(similarities_query[0])[-top_k:][::-1]

df.iloc[top_indices][["name", "salt_composition"]]

,name,salt_composition
7833,ANEKET 100 MG INJECTION,Ketamine (100mg)
102976,Gram SPN 500mg/250mg Injection,
68857,DUBAGEST 100 MG INJECTION,
36281,C Best 100 mg/200 mg Suppository,
102416,Gram SPN 250mg/125mg Injection,


In [30]:
df.iloc[top_indices][["name"]]

,name
7833,ANEKET 100 MG INJECTION
102976,Gram SPN 500mg/250mg Injection
68857,DUBAGEST 100 MG INJECTION
36281,C Best 100 mg/200 mg Suppository
102416,Gram SPN 250mg/125mg Injection


In [31]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve_top_k(query_text, embeddings, df, model, top_k=5):
    # Encode the query
    query_embedding = model.encode(
        query_text,
        convert_to_numpy=True
    )

    # Compute cosine similarity
    similarities = cosine_similarity(
        query_embedding.reshape(1, -1),
        embeddings
    )[0]

    # Get Top-K indices
    top_indices = np.argsort(similarities)[-top_k:][::-1]

    # Return Top-K medicine names
    return df.iloc[top_indices]["name"].tolist()

In [32]:
query = "I'm taking MeliGlyp M1 tablet SR as advised by my doctor."

retrieve_top_k(
    query,
    embeddings_name,
    df,
    model,
    top_k=5
)

['Meliglip M1 Tablet SR',
 'Meliglip M2 Tablet SR',
 'Melipride M2 2 mg/500 mg Tablet',
 'Glimbite-M1 Tablet SR',
 'Melovas 50mg Tablet XL']

In [33]:
import pandas as pd

metadata_df = pd.read_csv("metadata.csv")

In [34]:
print(metadata_df.shape)
print(metadata_df.columns)

(60, 7)
Index(['sample_id', 'audio_file', 'language', 'medicine_name',
       'stt_transcript', 'recovered_medicine', 'notes'],
      dtype='object')


In [35]:
predictions = []

In [36]:
for _, row in metadata_df.iterrows():

    transcript = row["stt_transcript"]

    top5 = retrieve_top_k(
        transcript,
        embeddings_name,
        df,
        model,
        top_k=5
    )

    predictions.append(top5)

In [37]:
len(predictions)

60

In [38]:
metadata_df["top5_predictions"] = predictions

In [39]:
metadata_df[["medicine_name", "top5_predictions"]].head()

,medicine_name,top5_predictions
0,Aneudox M Injection / Urigo 40mg Tablet,"[Anmox 1000mg/200mg Injection, Anwita 60mg Inj..."
1,Aneudox M Injection,"[Tusnox-A Cough Expectorant, Ambtic Cough Syru..."
2,Benfotor Injection,"[Benfotor Injection, Benuvo Injection, Benotin..."
3,Chemiclo SP 100mg/325mg/10mg Tablet,"[Migray Plus 40mg/10mg Tablet, Migray 40mg Tab..."
4,Cmerx Eye Drop / Exam 50mg Tablet ER,"[Simdrox 250mg Tablet, Flucalup 50mg Tablet, F..."


In [40]:
top1 = 0
top3 = 0
top5 = 0

total = len(metadata_df)

for _, row in metadata_df.iterrows():

    ground_truth = str(row["medicine_name"]).strip().lower()
    predictions = [p.strip().lower() for p in row["top5_predictions"]]

    if ground_truth == predictions[0]:
        top1 += 1

    if ground_truth in predictions[:3]:
        top3 += 1

    if ground_truth in predictions:
        top5 += 1

print(f"Top-1 Accuracy : {top1/total:.2%}")
print(f"Top-3 Accuracy : {top3/total:.2%}")
print(f"Top-5 Accuracy : {top5/total:.2%}")

Top-1 Accuracy : 3.33%
Top-3 Accuracy : 3.33%
Top-5 Accuracy : 3.33%


In [41]:
for i in range(10):
    print("=" * 80)
    print("Ground Truth :", metadata_df.loc[i, "medicine_name"])
    print("Predictions  :", metadata_df.loc[i, "top5_predictions"])

Ground Truth : Aneudox M Injection / Urigo 40mg Tablet
Predictions  : ['Anmox 1000mg/200mg Injection', 'Anwita 60mg Injection', 'She Cart 1000mg Injection', 'Michelle-AQ Injection 1ml', 'Ancy 1000mg Injection']
Ground Truth : Aneudox M Injection
Predictions  : ['Tusnox-A Cough Expectorant', 'Ambtic Cough Syrup', 'Deletus-BX Cough Expectorant', 'Febrinil-CC Cough & Cold Capsule', 'Asth 150mg Injection']
Ground Truth : Benfotor Injection
Predictions  : ['Benfotor Injection', 'Benuvo Injection', 'Benotin Plus Injection', 'Benvit Injection', 'Benpot 40 Injection']
Ground Truth : Chemiclo SP 100mg/325mg/10mg Tablet
Predictions  : ['Migray Plus 40mg/10mg Tablet', 'Migray 40mg Tablet SR', 'Synasma 100mg Injection', 'Panact 150mg Injection', 'Dayflozin M Forte 10mg/1000mg Tablet']
Ground Truth : Cmerx Eye Drop / Exam 50mg Tablet ER
Predictions  : ['Simdrox 250mg Tablet', 'Flucalup 50mg Tablet', 'Flurator 400mg Tablet', 'Flugesic 800mg Tablet', 'Flutacare 250mg Tablet']
Ground Truth : Damol 50m

In [42]:
#trying the exp 2

In [43]:
import re

def split_into_sentences(transcript):
    """
    Split transcript into sentences.
    Handles ., ?, ! and newlines.
    """
    if not isinstance(transcript, str):
        return []

    sentences = re.split(r'[.!?\n]+', transcript)

    # Remove empty sentences
    sentences = [s.strip() for s in sentences if s.strip()]

    return sentences

In [44]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve_top_k_best_sentence(
    transcript,
    embeddings,
    df,
    model,
    top_k=5
):
    sentences = split_into_sentences(transcript)

    if len(sentences) == 0:
        return []

    best_similarity = -1
    best_sentence = None

    for sentence in sentences:

        query_embedding = model.encode(
            sentence,
            convert_to_numpy=True
        )

        similarities = cosine_similarity(
            query_embedding.reshape(1, -1),
            embeddings
        )[0]

        sentence_best = similarities.max()

        if sentence_best > best_similarity:
            best_similarity = sentence_best
            best_sentence = sentence

    # Run retrieval again using ONLY the best sentence
    query_embedding = model.encode(
        best_sentence,
        convert_to_numpy=True
    )

    similarities = cosine_similarity(
        query_embedding.reshape(1, -1),
        embeddings
    )[0]

    top_indices = np.argsort(similarities)[-top_k:][::-1]

    return df.iloc[top_indices]["name"].tolist()

In [45]:
transcript = metadata_df.loc[0, "stt_transcript"]

retrieve_top_k_best_sentence(
    transcript,
    embeddings_name,
    df,
    model,
    top_k=5from google.colab import files

files.download("dense_retrieval_results_minilm.csv")
)

['Anti Pain 75mg Injection',
 'Pain Off 25mg Injection',
 'D Pain AQ 75mg Injection',
 'Re Pain 100mg Injection',
 'Anszol 20mg Injection']

In [46]:
##trying something new here

In [47]:
metadata_df.to_csv(
    "dense_retrieval_results_minilm.csv",
    index=False
)

In [48]:
from google.colab import files

files.download("dense_retrieval_results_minilm.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [49]:
df.to_csv(
    "medicine_dataset_with_embeddings.csv",
    index=False
)

In [52]:
from google.colab import files
import numpy as np

# Save the embeddings to .npy files
np.save("medicine_embeddings_minilm.npy", embeddings)
np.save("medicine_embeddings_name_minilm.npy", embeddings_name)

files.download("dense_retrieval_results_minilm.csv")
files.download("medicine_embeddings_minilm.npy")
files.download("medicine_embeddings_name_minilm.npy")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [53]:
metadata_df.to_csv(
    "dense_retrieval_results_minilm.csv",
    index=False,
    encoding="utf-8-sig"
)

In [54]:
from google.colab import files

files.download("dense_retrieval_results_minilm.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>